# **PYSPARK INTERVIEW QUESTIONS - SHASHANK KUMAR**

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window

**Q1 While ingesting customer data from an external source, you notice duplicate entries. How would you remove duplicates and retain only the latest entry based on a timestamp column?**

In [0]:

data = [("101", "2023-12-01", 100), ("101", "2023-12-02", 150), 
        ("102", "2023-12-01", 200), ("102", "2023-12-02", 250)]
columns = ["product_id", "date", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

product_id,date,sales
101,2023-12-01,100
101,2023-12-02,150
102,2023-12-01,200
102,2023-12-02,250


**Solution**

In [0]:
#casting date column data to date data type

df = df.withColumn('date', col('date').cast(DateType()))

df.display()

product_id,date,sales
101,2023-12-01,100
101,2023-12-02,150
102,2023-12-01,200
102,2023-12-02,250


In [0]:
#Drop Duplicate
df.orderBy('product_id','date',ascending=[1,0]).dropDuplicates(subset=['product_id']).display()

product_id,date,sales
101,2023-12-02,150
102,2023-12-02,250


**2. While processing data from multiple files with inconsistent schemas, you need to merge them into a single DataFrame. How would you handle this inconsistency in PySpark?**

**Solution**

In [0]:
df=spark.read.format('parquet')\
    .option('mergeSchema',True)\
        .load('file_path')

---------------------------------------------------------------------------
IllegalArgumentException                  Traceback (most recent call last)
File <command-3040968968153621>:1
----> 1 df=spark.read.format('parquet')\
      2     .option('mergeSchema',True)\
      3         .load('file_path')

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/sql/readwriter.py:302, in DataFrameReader.load(self, path, format, schema, **options)
    300 self.options(**options)
    301 if isinstance(path, str):
--> 302     return self._df(self._jreader.load(path))
    303 elif path is not None:
    304     if type(path) != list:

File /databr

**4. You are working with a real-time data pipeline, and you notice missing values in your streaming data Column - Category. How would you handle null or missing values in such a scenario?**

**df_stream = spark.readStream.schema("id INT, value STRING").csv("path/to/stream")**

In [0]:
df= df.fillna({'Category':'N/a'})

**5. You need to calculate the total number of actions performed by users in a system. How would you calculate the top 2 most active users based on this information?**

In [0]:
data = [("user1", 5), ("user2", 8), ("user3", 2), ("user4", 10), ("user2", 3)]
columns = ["user_id", "actions"]

df = spark.createDataFrame(data, columns)
df.display()

user_id,actions
user1,5
user2,8
user3,2
user4,10
user2,3


In [0]:
df=df.groupBy(col('user_id')).\
    agg(sum(col('actions')).alias('total_actions')).\
        orderBy(col('total_actions').desc()).\
            limit(2)
df.display()

user_id,total_actions
user2,11
user4,10


**6. While processing sales transaction data, you need to identify the most recent transaction for each customer. How would you approach this task?**

In [0]:
data = [("cust1", "2023-12-01", 100), ("cust2", "2023-12-02", 150),
        ("cust1", "2023-12-03", 200), ("cust2", "2023-12-04", 250)]
columns = ["customer_id", "transaction_date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

customer_id,transaction_date,sales
cust1,2023-12-01,100
cust2,2023-12-02,150
cust1,2023-12-03,200
cust2,2023-12-04,250


In [0]:
df=df.withColumn('transaction_date', col('transaction_date').cast(dataType=DateType()))
df.display()

customer_id,transaction_date,sales
cust1,2023-12-01,100
cust2,2023-12-02,150
cust1,2023-12-03,200
cust2,2023-12-04,250


###### method 1


In [0]:
df_recent=df.select('customer_id','transaction_date','sales').groupBy('customer_id').agg(max('transaction_date').alias("last_transaction_day"))
df_recent.display()


customer_id,last_transaction_day
cust1,2023-12-03
cust2,2023-12-04


In [0]:
df_alias = df.alias("df")
df_recent_alias = df_recent.alias("df_recent")

new_df = df_alias.join(df_recent_alias, 
                       (df_alias["customer_id"] == df_recent_alias["customer_id"]) & 
                       (df_alias["transaction_date"] == df_recent_alias["last_transaction_day"]), 
                       "inner") \
                 .select(df_alias["customer_id"], df_alias["transaction_date"], df_alias["sales"])

new_df.display()

customer_id,transaction_date,sales
cust1,2023-12-03,200
cust2,2023-12-04,250


###### method 2

In [0]:
windospace=Window.partitionBy('customer_id').orderBy(col('transaction_date').desc())
new_df=df.withColumn('Rank', row_number().over(windospace)).filter(col('Rank')==1).drop('rank')
new_df.display()

customer_id,transaction_date,sales
cust1,2023-12-03,200
cust2,2023-12-04,250


**7. You need to identify customers who haven’t made any purchases in the last 30 days. How would you filter such customers?**

In [0]:
data = [("cust1", "2025-03-01"), ("cust2", "2024-11-20"), ("cust3", "2025-02-25")]
columns = ["customer_id", "last_purchase_date"]

df = spark.createDataFrame(data, columns)

df.display()

customer_id,last_purchase_date
cust1,2025-03-01
cust2,2024-11-20
cust3,2025-02-25


In [0]:
df=df.withColumn('last_purchase_date', col('last_purchase_date').cast(DateType()))


In [0]:
df_filtered =df.filter(datediff(current_date(),col('last_purchase_date'))>30)
df_filtered.display()

customer_id,last_purchase_date
cust2,2024-11-20


**8. While analyzing customer reviews, you need to identify the most frequently used words in the feedback. How would you implement this?**

In [0]:
data = [("customer1", "The product is great"), ("customer2", "Great product, fast delivery"), ("customer3", "Not bad, could be better")]
columns = ["customer_id", "feedback"]

df = spark.createDataFrame(data, columns)

df.display()

customer_id,feedback
customer1,The product is great
customer2,"Great product, fast delivery"
customer3,"Not bad, could be better"


In [0]:
new_df=df.withColumn('feedback',lower('feedback')).withColumn('feedback',explode(split(col('feedback'),'[,\\s]+')))

In [0]:
new_df1=new_df.groupBy('feedback' ).agg(count("feedback").alias('count'))


In [0]:
max_count= new_df1.agg(max('count')).collect()[0][0]
print(max_count)

2


In [0]:
new_df1.filter(col('count')==max_count).display()

feedback,count
great,2
product,2


**9. You need to calculate the cumulative sum of sales over time for each product. How would you approach this?**

In [0]:
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-02", 200),
        ("product1", "2023-12-03", 150), ("product2", "2023-12-04", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

product_id,date,sales
product1,2023-12-01,100
product2,2023-12-02,200
product1,2023-12-03,150
product2,2023-12-04,250


In [0]:
df=df.withColumn('date', col('date').cast(DateType()))
df.display()

product_id,date,sales
product1,2023-12-01,100
product2,2023-12-02,200
product1,2023-12-03,150
product2,2023-12-04,250


In [0]:
windospace=Window.partitionBy('product_id').orderBy('date')


In [0]:
new_df= df.withColumn('cumsum',sum(col('sales')).over(windospace))
new_df.display()

product_id,date,sales,cumsum
product1,2023-12-01,100,100
product1,2023-12-03,150,250
product2,2023-12-02,200,200
product2,2023-12-04,250,450


**10. While preparing a data pipeline, you notice some duplicate rows in a dataset. How would you remove the duplicates without affecting the original order?**

In [0]:
data = [("John", 25), ("Jane", 30), ("John", 25), ("Alice", 22)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)
df.display()

name,age
John,25
Jane,30
John,25
Alice,22


In [0]:
df.dropDuplicates().display()

name,age
John,25
Jane,30
Alice,22


**11. You are working with user activity data and need to calculate the average session duration per user. How would you implement this?**

In [0]:
data = [("user1", "2023-12-01", 50), ("user1", "2023-12-02", 60), 
        ("user2", "2023-12-01", 45), ("user2", "2023-12-03", 75)]
columns = ["user_id", "session_date", "duration"]
df = spark.createDataFrame(data, columns)

df.display()

user_id,session_date,duration
user1,2023-12-01,50
user1,2023-12-02,60
user2,2023-12-01,45
user2,2023-12-03,75


In [0]:
df.groupBy(col('user_id')).agg(avg('duration')).display()

user_id,avg(duration)
user1,55.0
user2,60.0


**12. While analyzing sales data, you need to find the product with the highest sales for each month. How would you accomplish this?**

In [0]:
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-01", 150), 
        ("product1", "2023-12-02", 200), ("product2", "2024-12-02", 250),
        ("product1", "2024-11-02", 200), ("product2", "2023-11-02", 250),
        ("product1", "2024-10-02", 500), ("product2", "2024-01-02", 350)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

product_id,date,sales
product1,2023-12-01,100
product2,2023-12-01,150
product1,2023-12-02,200
product2,2024-12-02,250
product1,2024-11-02,200
product2,2023-11-02,250
product1,2024-10-02,500
product2,2024-01-02,350


In [0]:
df=df.withColumn('date',col('date').cast(DateType()))
df.display()

product_id,date,sales
product1,2023-12-01,100
product2,2023-12-01,150
product1,2023-12-02,200
product2,2024-12-02,250
product1,2024-11-02,200
product2,2023-11-02,250
product1,2024-10-02,500
product2,2024-01-02,350


In [0]:
df_group=df.groupBy(year('date').alias('year'),month('date').alias('month'),col('product_id')).agg(sum('sales').alias('sum'))

In [0]:
df_group.display()

year,month,product_id,sum
2023,12,product1,300
2023,12,product2,150
2024,12,product2,250
2024,11,product1,200
2023,11,product2,250
2024,10,product1,500
2024,1,product2,350


In [0]:
windospace=Window.partitionBy(col('year'),col('month')).orderBy(col('sum').desc())

In [0]:
df_part=df_group.withColumn('rank', row_number().over(windospace))

In [0]:
df_part.display()

year,month,product_id,sum,rank
2023,11,product2,250,1
2023,12,product1,300,1
2023,12,product2,150,2
2024,1,product2,350,1
2024,10,product1,500,1
2024,11,product1,200,1
2024,12,product2,250,1


In [0]:
df_part.filter(col('rank')==1).display()

year,month,product_id,sum,rank
2023,11,product2,250,1
2023,12,product1,300,1
2024,1,product2,350,1
2024,10,product1,500,1
2024,11,product1,200,1
2024,12,product2,250,1


**13. You are working with a large Delta table that is frequently updated by multiple users. The data is stored in partitions, and sometimes updates can cause inconsistent reads due to concurrent transactions. How would you ensure ACID compliance and avoid data corruption in PySpark?**

In [0]:
#new data

df =spark.read.format('parquet').load('path')

from delta.tables import DeltaTable

delta_table = DeltaTable.forPath('path')

delta_table.alias('trg').merge(df.alias('src'),'src.id==trg.id')\
    .whenNotMatchedInsertAll()\
        .whenMatchedUpdateAll()\
            .execute()

**14. You need to process a large dataset stored in PARQUET format and ensure that all columns have the right schema (Almost). How would you do this?**

In [0]:
df.read.format('parquet')\
    .option('inferSchema',True)\
        .load('path')

**15. You are reading a CSV file and need to handle corrupt records gracefully by skipping them. How would you configure this in PySpark?**
drop corrupted data


In [0]:
df = spark.read.option("mode", "DROPMALFORMED").json("data.json")
df.show()


**PERMISSIVE** (Default):	Keeps bad records, fills missing/invalid values with null.

**DROPMALFORMED**:	Drops malformed rows, continues processing.

**FAILFAST**:	Stops execution immediately on error.

**22. You have a dataset containing the names of employees and their departments. You need to find the department with the most employees.**

In [0]:
data = [("Alice", "HR"), ("Bob", "Finance"), ("Charlie", "HR"), ("David", "Engineering"), ("Eve", "Finance")]
columns = ["employee_name", "department"]

df = spark.createDataFrame(data, columns)
df.display()

employee_name,department
Alice,HR
Bob,Finance
Charlie,HR
David,Engineering
Eve,Finance


In [0]:
df1=df.groupBy(col('department')).agg(count('employee_name').alias('emp_count')).\
    orderBy(col('emp_count').desc())


In [0]:
max_count=df1.agg(max('emp_count')).collect()[0][0]
df2=df1.filter(col('emp_count')==max_count)
df2.display()

department,emp_count
HR,2
Finance,2


**23. While processing sales data, you need to classify each transaction as either 'High' or 'Low' based on its amount. How would you achieve this using a when condition**

In [0]:
data = [("product1", 100), ("product2", 300), ("product3", 50)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

product_id,sales
product1,100
product2,300
product3,50


In [0]:
df1=df.withColumn('flag', lit(when(col('sales')>=100,'high').otherwise('low')))

In [0]:
df1.display()

product_id,sales,flag
product1,100,high
product2,300,high
product3,50,low


**24. While analyzing a large dataset, you need to create a new column that holds a timestamp of when the record was processed. How would you implement this and what can be the best USE CASE?**

In [0]:
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

product_id,sales
product1,100
product2,200
product3,300


In [0]:
df.withColumn('Timestamp',lit(current_timestamp())).display()

product_id,sales,Timestamp
product1,100,2025-03-14T15:16:54.700+0000
product2,200,2025-03-14T15:16:54.700+0000
product3,300,2025-03-14T15:16:54.700+0000


**25. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it. How would you achieve this?**

In [0]:
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

product_id,sales
product1,100
product2,200
product3,300


In [0]:
df.createOrReplaceTempView('sales_detail')
result=spark.sql("select * from sales_detail")
result.display()

product_id,sales
product1,100
product2,200
product3,300


**26. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it (FROM DIFFERENT NOTEBOOKS AS WELL)?**

In [0]:
df.createOrReplaceGlobalTempView('global_sales_dt')


In [0]:
spark.sql('select * from global_temp.global_sales_dt')


Out[10]: DataFrame[product_id: string, sales: bigint]

**27. You need to query data from a PySpark DataFrame using SQL, but the data includes a nested structure. How would you flatten the data for easier querying?**

In [0]:
data = [("product1", {"price": 100, "quantity": 2}), 
        ("product2", {"price": 200, "quantity": 3})]
columns = ["product_id", "product_info"]

df = spark.createDataFrame(data, columns)
df.display()

product_id,product_info
product1,"Map(price -> 100, quantity -> 2)"
product2,"Map(price -> 200, quantity -> 3)"


In [0]:
df_flattened = df.select(
    col("product_id"),
    col("product_info").getItem("price").alias("price"),
    col("product_info").getItem("quantity").alias("quantity")
)

df_flattened.show()

+----------+-----+--------+
|product_id|price|quantity|
+----------+-----+--------+
|  product1|  100|       2|
|  product2|  200|       3|
+----------+-----+--------+



**28. You are ingesting data from an external API in JSON format where the schema is inconsistent. How would you handle this situation to ensure a robust pipeline?**

In [0]:
df.read.format('json').option('mergeSchema',True).

**29. While reading data from Parquet, you need to optimize performance by partitioning the data based on a column. How would you implement this?**

In [0]:
df.write.format('parquet').mode('append').partitionBy('category').save('path')

**30. You are working with a large dataset in Parquet format and need to ensure that the data is written in an optimized manner with proper compression. How would you accomplish this?**

In [0]:
df.write.format('parquet').option('compression','snappy')

**31. Your company uses a large-scale data pipeline that reads from Delta tables and processes data using complex aggregations. However, performance is becoming an issue due to the growing dataset size. How would you optimize the performance of the pipeline?**

In [0]:
%sql
optimize tabledelta zorder by ("order_date")

**43. You are processing sales data. Group by product categories and create a list of all product names in each category.**

In [0]:
data = [("Electronics", "Laptop"), ("Electronics", "Smartphone"), ("Furniture", "Chair"), ("Furniture", "Table")]
columns = ["category", "product"]
df = spark.createDataFrame(data, columns)
df.display()

category,product
Electronics,Laptop
Electronics,Smartphone
Furniture,Chair
Furniture,Table


In [0]:
df_list= df.groupBy('category').agg(collect_list('product'))

In [0]:
df_list.display()

category,collect_list(product)
Electronics,"List(Laptop, Smartphone)"
Furniture,"List(Chair, Table)"


**44. You are analyzing orders. Group by customer IDs and list all unique product IDs each customer purchased.**

In [0]:
data = [(101, "P001"), (101, "P002"), (102, "P001"), (101, "P001")]
columns = ["customer_id", "product_id"]
df = spark.createDataFrame(data, columns)
df.display()

customer_id,product_id
101,P001
101,P002
102,P001
101,P001


In [0]:
df_set=df.groupBy('customer_id').agg(collect_set('product_id'))
df_set.display()

customer_id,collect_set(product_id)
101,"List(P002, P001)"
102,List(P001)


**45. For customer records, combine first and last names only if the email address exists.**

In [0]:
data = [("John", "Doe", "john.doe@example.com"), ("Jane", "Smith", None)]
columns = ["first_name", "last_name", "email"]
df = spark.createDataFrame(data, columns)
df.display()

first_name,last_name,email
John,Doe,john.doe@example.com
Jane,Smith,null


In [0]:
df.withColumn(
    'Full Name',
    when(
        col('email').isNotNull(),\
        concat_ws(' ',col('first_name'),col('last_name'))
        )
    ).display()

first_name,last_name,email,Full Name
John,Doe,john.doe@example.com,John Doe
Jane,Smith,null,null


In [0]:
df.withColumn('name', 
    when(col('email').isNotNull(),concat(col('first_name'),lit(' '),col('last_name')))
              ).display()

first_name,last_name,email,name
John,Doe,john.doe@example.com,John Doe
Jane,Smith,null,null


**46. You have a DataFrame containing customer IDs and a list of their purchased product IDs. Calculate the number of products each customer has purchased.**

In [0]:
data = [
    (1, ["prod1", "prod2", "prod3"]),
    (2, ["prod4"]),
    (3, ["prod5", "prod6"]),
]
myschema = "customer_id INT ,product_ids array<STRING>"

df = spark.createDataFrame(data, myschema)
df.display()

customer_id,product_ids
1,"List(prod1, prod2, prod3)"
2,List(prod4)
3,"List(prod5, prod6)"


In [0]:
df.withColumn('count',size(col('product_ids'))).display()

customer_id,product_ids,count
1,"List(prod1, prod2, prod3)",3
2,List(prod4),1
3,"List(prod5, prod6)",2


**47. You have employee IDs of varying lengths. Ensure all IDs are 6 characters long by padding with leading zeroes.**

In [0]:
data = [
    ("1",),
    ("123",),
    ("4567",),
]
schema = ["employee_id"]

df = spark.createDataFrame(data, schema)
df.display()

employee_id
1
123
4567


In [0]:
df.withColumn('employee_id', lpad('employee_id',6,'0')).display()

employee_id
000001
000123
004567


**48. You need to validate phone numbers by checking if they start with "91"**

In [0]:
data = [
    ("911234567890",),
    ("811234567890",),
    ("912345678901",),
]
schema = ["phone_number"]

df = spark.createDataFrame(data, schema)
df.display()

phone_number
911234567890
811234567890
912345678901


In [0]:
df.filter(substring(col('phone_number'),1,2)=='91').display()

phone_number
911234567890
912345678901


In [0]:
df.withColumn('phone_number', col('phone_number').startswith('91')).display()

phone_number
true
false
true


**49. You have a dataset with courses taken by students. Calculate the average number of courses per student.**

In [0]:
data = [
    (1, ["Math", "Science"]),
    (2, ["History"]),
    (3, ["Art", "PE", "Biology"]),
]
schema = ["student_id", "courses"]

df = spark.createDataFrame(data, schema)
df.display()

student_id,courses
1,"List(Math, Science)"
2,List(History)
3,"List(Art, PE, Biology)"


In [0]:
df.agg(avg(size('courses'))).display()

avg(size(courses))
2.0


**50. You have a dataset with primary and secondary contact numbers. Use the primary number if available; otherwise, use the secondary number.**

In [0]:
data = [
    (None, "1234567890"),
    ("9876543210", None),
    ("7894561230", "4567891230"),
]
schema = ["primary_contact", "secondary_contact"]

df = spark.createDataFrame(data, schema)
df.display()

primary_contact,secondary_contact
null,1234567890
9876543210,null
7894561230,4567891230


In [0]:
df.withColumn('contact', coalesce(col('primary_contact'),col('secondary_contact'))).display()

primary_contact,secondary_contact,contact
null,1234567890,1234567890
9876543210,null,9876543210
7894561230,4567891230,7894561230


In [0]:
df.withColumn('use_number',
    when(col('primary_contact').isNotNull(),col('primary_contact')).\
        otherwise(col('secondary_contact'))
                               ).display()

primary_contact,secondary_contact,use_number
null,1234567890,1234567890
9876543210,null,9876543210
7894561230,4567891230,7894561230


**51. You are categorizing product codes based on their lengths. If the length is 5, label it as "Standard"; otherwise, label it as "Custom".**

In [0]:
data = [
    ("prod1",),
    ("prd234",),
    ("pr9876",),
]
schema = ["product_code"]

df = spark.createDataFrame(data, schema)
df.display()

product_code
prod1
prd234
pr9876


In [0]:
df.withColumn('label',
              when(
                  length(col('product_code'))==5,'Standerd'
                  ).otherwise('Custom')).display()

product_code,label
prod1,Standerd
prd234,Custom
pr9876,Custom
